# EasyMLOps  
  
## 介绍   
`EasyMLOps`包以`Pipline`的方式构建建模任务，可直接进行模型训练、预测(离线，在线)，测试(离线在线预测一致性、预测性能)、特征存储、监控、分析等功能，通过外套一层Flask或FastApi即可直接部署生产

## 安装
```bash
pip install easymlops-版本号
```  

## 使用

In [1]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
print(data.head(5).to_markdown())

|    |   PassengerId |   Survived |   Pclass | Name                                                | Sex    |   Age |   SibSp |   Parch | Ticket           |    Fare | Cabin   | Embarked   |
|---:|--------------:|-----------:|---------:|:----------------------------------------------------|:-------|------:|--------:|--------:|:-----------------|--------:|:--------|:-----------|
|  0 |             1 |          0 |        3 | Braund, Mr. Owen Harris                             | male   |    22 |       1 |       0 | A/5 21171        |  7.25   | nan     | S          |
|  1 |             2 |          1 |        1 | Cumings, Mrs. John Bradley (Florence Briggs Thayer) | female |    38 |       1 |       0 | PC 17599         | 71.2833 | C85     | C          |
|  2 |             3 |          1 |        3 | Heikkinen, Miss. Laina                              | female |    26 |       0 |       0 | STON/O2. 3101282 |  7.925  | nan     | S          |
|  3 |             4 |          1 |        1 | Fut

In [2]:
# 拆分训练测试
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### 表格型任务

In [3]:
from easymlops import TablePipeLine
from easymlops.table.strategy import LGBMRegressionLayers
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.decomposition import *
from easymlops.table.classification import *
from easymlops.table.ensemble import *

In [4]:
table = TablePipeLine()
table.pipe(FixInput()) \
.pipe(FillNa()) \
.pipe(LGBMRegressionLayers(y=y_train, cols_layer=[["Name", "Ticket"],["Embarked", "Cabin", "Sex"]],
                           embedding_target_encoding_params_layer=[
                               {"cols":["Name","Ticket"]},
                               {"cols":["Embarked", "Cabin", "Sex"]}],use_faster_predictor=True))
table.fit(x_train)

In [5]:
# table = TablePipeLine()
# table.pipe(FixInput()) \
# .pipe(FillNa()) \
# .pipe(LGBMRegressionLayers(y=y_train, cols_layer=[["Name", "Ticket"],["Embarked", "Cabin", "Sex"]],
#                            embedding_target_encoding_params_layer=[
#                                {"cols":["Name","Ticket"]},
#                                {"cols":["Embarked", "Cabin", "Sex"]}],use_faster_predictor=True))
# table.fit(x_train)

In [6]:
print(table.transform(x_test).head(5).to_markdown())

|     |     pred |
|----:|---------:|
| 500 | 0.13367  |
| 501 | 0.853646 |
| 502 | 0.853646 |
| 503 | 0.611105 |
| 504 | 0.851839 |


In [7]:
table.transform_single(x_test.to_dict("record")[0])

{'pred': 0.13367006480398747}

In [8]:
# table.auto_test(x_test[:10])

In [9]:
table.save("demo.pkl")

In [10]:
new_table=TablePipeLine()
new_table.load("demo.pkl")

In [11]:
print(new_table.transform(x_test).head(5).to_markdown())

|     |     pred |
|----:|---------:|
| 500 | 0.13367  |
| 501 | 0.853646 |
| 502 | 0.853646 |
| 503 | 0.611105 |
| 504 | 0.851839 |


In [12]:
new_table.transform_single(x_test.to_dict("record")[0])

{'pred': 0.13367006480398747}

In [13]:
new_table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.strategy.lgbm_regression_layer.LGBMRegressionLayers'>) module check [transform] complete,speed:[1.56ms]/it,cpu:[0%],memory:[0K]

#########################################################################################
 2.空值测试:check_null_value                                                           
 null_values=[None, np.nan, "null", "NULL", "nan", "NaN", "", "none", "None", " "](默认)
########################################################################################
column:[PassengerId] check [null